# Demonstration and Evaluation of `fisher_single_sample`

This notebook demonstrates the usage and evaluates the properties of the covariance hypothesis testing method `fisher_single_sample` from `covtest.methods.hypothesis_identity`.

## Hypothesis Tested
We test the null hypothesis:
$$H_0: \Sigma = I_p$$
against the alternative hypothesis:
$$H_1: \Sigma \neq I_p$$
where $\Sigma$ is the population covariance matrix of the data, and $I_p$ is the identity matrix of dimension $p$.

## Method Description
Fisher (2012) LRT/Trace test for single-sample covariance structure.

## Notebook Contents
1. **Setup and Imports**
2. **Basic Usage**
3. **P-value Calibration Under the Null**: Checking uniformity of p-values under different distributions (Gaussian, Student-t, Laplace).
4. **Power Analysis Under Alternatives**: Power curves across different sample sizes ($n$), dimensions ($p$), and alternative covariance types (spiked, Toeplitz).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Ensure project root is in the path
sys.path.insert(0, os.path.abspath('..'))

from covtest.methods.hypothesis_identity import fisher_single_sample
from covtest.simulation.generate_data import generate_heavy_tailed_samples
from covtest.simulation.generate_covariances import (
    generate_toeplitz_cov,
    generate_spiked_covariance
)
from covtest.diagnostics.evaluate_pvalues import analyze_pvalues
from covtest.plotting.null import plot_pvalue_diagnostics_grid
from covtest.plotting.alternative import plot_power_curve, plot_power_heatmap

# Set up plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
rng = np.random.default_rng(42)


## 2. Basic Usage

Let's run a single test on synthetic data generated under the null hypothesis ($H_0: \Sigma = I_p$).


In [ ]:
n, p = 100, 50
null_cov = np.eye(p)

# Generate multivariate normal data under the null
X = generate_heavy_tailed_samples(null_cov, n, dist_type="normal", rng=rng)

# Run the test
result = fisher_single_sample(X, Sigma=null_cov)
print("Test Results:")
for k, v in result.items():
    print(f"  {k}: {v}")


## 3. P-value Calibration Under the Null

Under the null hypothesis, the p-values computed by the test should theoretically be uniformly distributed over $[0, 1]$. We will evaluate this calibration by running $M = 300$ simulations under three different scenarios:
- **Scenario A**: Multivariate Normal distribution (standard).
- **Scenario B**: Multivariate Student-t distribution (heavy-tailed, df=3), which violates standard normality assumptions.
- **Scenario C**: Multivariate Laplace distribution.


In [ ]:
M = 300
n, p = 100, 30
null_cov = np.eye(p)

scenarios = {
    "Normal": ("normal", {}),
    "Student-t (df=3)": ("t", {"df": 3}),
    "Laplace": ("laplace", {})
}

pvals_dict = {}

print("Running null simulations...")
for name, (dist, opts) in scenarios.items():
    pvals = []
    for _ in range(M):
        X = generate_heavy_tailed_samples(null_cov, n, dist_type=dist, options=opts, rng=rng)
        res = fisher_single_sample(X, Sigma=null_cov)
        pvals.append(res["p_value"])
    pvals_dict[name] = np.array(pvals)
    print(f"  Completed {name}")


### Diagnostic Analysis of Null P-values

We use the package's `analyze_pvalues` diagnostic utility to check the uniformity of p-values for each scenario. A well-calibrated test under $H_0$ will exhibit:
- A Kolmogorov-Smirnov p-value > 0.05 (fail to reject uniformity).
- A genomic inflation factor $\lambda_{{GC}} \approx 1$.
- Storey's $\pi_0 \approx 1$.


In [ ]:
for name, pvals in pvals_dict.items():
    diagnostics = analyze_pvalues(pvals)
    print(f"=== {name} Scenario Diagnostics ===")
    print(f"  KS Test p-value: {diagnostics['ks']['pval']:.4f}")
    print(f"  Anderson-Darling stat: {diagnostics['ad']['stat']:.4f}")
    print(f"  Genomic Inflation Factor (lambda): {diagnostics['inflation_factor']:.4f}")
    print(f"  Storey's pi0: {diagnostics['storey_pi0']:.4f}")
    print()


### Visualizing P-value Calibration
Let's plot the ECDF, histogram, and QQ-plot for each scenario to visually assess the uniformity of the null p-values.


In [ ]:
for name, pvals in pvals_dict.items():
    fig = plot_pvalue_diagnostics_grid(pvals)
    fig.suptitle(f"P-value Calibration Grid - {name} Null", fontsize=16)
    plt.tight_layout()
    plt.show()


## 4. Power Analysis Under Alternatives

We evaluate the statistical power of the `fisher_single_sample` test. Power is the probability of correctly rejecting the null hypothesis when the alternative is true (measured here as the proportion of p-values $< 0.05$).

We consider two alternative covariance structures:
1. **Spiked Covariance Matrix**: A few eigenvalues are spiked (inflated) to simulate a dominant variance structure.
2. **Toeplitz (AR(1)) Covariance Matrix**: An autoregressive covariance structure where correlation decays exponentially with distance.

### Power vs. Alternative Strength
We vary the strength of the alternative covariance structure and record the rejection rates. For both curves, the parameter value at the start (1.0 for spike, 0.0 for Toeplitz rho) corresponds to the null covariance, where power should be approximately $\alpha = 0.05$.


In [ ]:
M_alt = 100
n, p = 100, 30

# Spiked alternative
spike_values = np.array([1.0, 1.5, 2.0, 3.0, 4.5, 6.0])
pvals_spiked = []

for spike in spike_values:
    col_pvals = []
    cov_matrix = generate_spiked_covariance(p, spike_eigenvalue=spike, num_spikes=1, rng=rng) if spike > 1.0 else np.eye(p)
    for _ in range(M_alt):
        X = generate_heavy_tailed_samples(cov_matrix, n, dist_type="normal", rng=rng)
        res = fisher_single_sample(X, Sigma=np.eye(p))
        col_pvals.append(res["p_value"])
    pvals_spiked.append(col_pvals)

pvals_spiked = np.array(pvals_spiked).T


In [ ]:
# Toeplitz alternative
rho_values = np.array([0.0, 0.1, 0.2, 0.3, 0.4, 0.5])
pvals_toeplitz = []

for rho in rho_values:
    col_pvals = []
    cov_matrix = generate_toeplitz_cov(p, rho) if rho > 0.0 else np.eye(p)
    for _ in range(M_alt):
        X = generate_heavy_tailed_samples(cov_matrix, n, dist_type="normal", rng=rng)
        res = fisher_single_sample(X, Sigma=np.eye(p))
        col_pvals.append(res["p_value"])
    pvals_toeplitz.append(col_pvals)

pvals_toeplitz = np.array(pvals_toeplitz).T


In [ ]:
# Plotting spiked power curve
print("Plotting power curve for Spiked Covariance:")
plot_power_curve(pvals_spiked, spike_values, alpha=0.05)


In [ ]:
# Plotting Toeplitz power curve
print("Plotting power curve for Toeplitz Covariance:")
plot_power_curve(pvals_toeplitz, rho_values, alpha=0.05)


### Power Heatmap Over Sample Sizes ($n$) and Dimensions ($p$)
We now analyze how power varies over a grid of sample sizes and dimensions under a fixed spiked alternative covariance.


In [ ]:
sample_sizes = np.array([30, 60, 120])
dimensions = np.array([10, 30, 80])
pvals_grid = []
spike_val = 3.5

print("Running grid simulations...")
for n_val in sample_sizes:
    row_pvals = []
    for p_val in dimensions:
        cell_pvals = []
        cov_matrix = generate_spiked_covariance(p_val, spike_eigenvalue=spike_val, num_spikes=1, rng=rng)
        for _ in range(50):
            X = generate_heavy_tailed_samples(cov_matrix, n_val, dist_type="normal", rng=rng)
            res = fisher_single_sample(X, Sigma=np.eye(p_val))
            cell_pvals.append(res["p_value"])
        row_pvals.append(cell_pvals)
    pvals_grid.append(row_pvals)

pvals_grid = np.transpose(np.array(pvals_grid), (2, 0, 1))

print("Plotting power heatmap over sample sizes (n) and dimensions (p):")
plot_power_heatmap(pvals_grid, sample_sizes, dimensions, alpha=0.05)
